In [5]:
# LIBRARIES FOR FEATURE ENGINEERING

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier

In [11]:
cleaned_customer_transaction_data=pd.read_csv(r"C:\Users\user\Downloads\Fraudulent_Transaction_Detection_for_Finlora_Company\Finlora_dataset\artifacts\EDA_Data.csv")

Threshold-based features were created based on key behavioral and risk indicators identified during the exploratory analysis. These include low KYC verification levels, location mismatches between the registered country and IP address and high risk transaction characteristics such as high transaction amounts, elevated IP risk scores, and low device trust scores. These features were identified as high_risk thresholds due to their significantly elevated fraud rates. These thresholds were converted into binary risk indicators to enhance the model's ability to detect suspicious transactions.

In [13]:
# creating a threshold based features from the following risk_signal
cleaned_customer_transaction_data['timestamp'] = pd.to_datetime(cleaned_customer_transaction_data['timestamp'])
cleaned_customer_transaction_data['late_night_hour']=((cleaned_customer_transaction_data['hour']>= 3) & (cleaned_customer_transaction_data['hour'] <= 7)).astype(int)
cleaned_customer_transaction_data['amount_high']=(cleaned_customer_transaction_data['amount_usd']> 1000).astype(int)
cleaned_customer_transaction_data['high_ip_risk'] =(cleaned_customer_transaction_data['ip_risk_score'] > 0.8).astype(int)
cleaned_customer_transaction_data['low_device_trust'] =(cleaned_customer_transaction_data['device_trust_score']<0.5).astype(int)
cleaned_customer_transaction_data['new_account']= (cleaned_customer_transaction_data['account_age_days']>=30)& (cleaned_customer_transaction_data['account_age_days']<=90).astype(int)
cleaned_customer_transaction_data['very_new_account']=(cleaned_customer_transaction_data['account_age_days']<30).astype(int)
cleaned_customer_transaction_data['velocity_spike']=(cleaned_customer_transaction_data['txn_velocity_1h']>=3).astype(int)

high_risk_signal_features = cleaned_customer_transaction_data[['late_night_hour', 'amount_high', 'high_ip_risk', 'low_device_trust', 'new_account', 'very_new_account', 'velocity_spike']]
high_risk_signal_features.head()


,late_night_hour,amount_high,high_ip_risk,low_device_trust,new_account,very_new_account,velocity_spike
0,0,0,0,0,False,0,0
1,0,0,0,1,False,0,0
2,0,0,0,0,False,0,0
3,0,0,0,0,False,0,0
4,0,0,0,0,False,0,0


In [14]:
list(cleaned_customer_transaction_data.columns)

['Unnamed: 0',
 'transaction_id',
 'customer_id',
 'timestamp',
 'home_country',
 'source_currency',
 'dest_currency',
 'channel',
 'amount_src',
 'amount_usd',
 'fee',
 'exchange_rate_src_to_dest',
 'device_id',
 'new_device',
 'ip_address',
 'ip_country',
 'location_mismatch',
 'ip_risk_score',
 'kyc_tier',
 'account_age_days',
 'device_trust_score',
 'chargeback_history_count',
 'risk_score_internal',
 'txn_velocity_1h',
 'txn_velocity_24h',
 'corridor_risk',
 'is_fraud',
 'hour',
 'day_of_week',
 'is weekend',
 'month',
 'account_age_group',
 'device_trust_score_bucket',
 'ip_risk_score_bucket',
 'amount_usd_bucket',
 'late_night_hour',
 'amount_high',
 'high_ip_risk',
 'low_device_trust',
 'new_account',
 'very_new_account',
 'velocity_spike']

### Feature Selection

In [ ]:
# Dropping all temporaary buskets columm 
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(['account_age_bucket','device_trust_score_bucket','ip_risk_score_bucket','amount_usd_bucket'], axis=1)
# Also i will be dropping all identifiers colum (Ids)
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(['transaction_id','customer_id','device_id','ip_address'],axis=1)
# Dropping some irrelevant variables =
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(['chargeback_history_count','exchange_rate_src_to_dest','Unnamed: 0'],axis=1)

### Defining Categorical feautres 

In [16]:
categorical_features = cleaned_customer_transaction_data.select_dtypes(include=['object','bool']).columns
categorical_features

Index(['transaction_id', 'customer_id', 'home_country', 'source_currency',
       'dest_currency', 'channel', 'amount_src', 'device_id', 'new_device',
       'ip_address', 'ip_country', 'location_mismatch', 'kyc_tier',
       'account_age_group', 'device_trust_score_bucket',
       'ip_risk_score_bucket', 'amount_usd_bucket', 'new_account'],
      dtype='object')

In [17]:
cleaned_customer_transaction_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11006 entries, 0 to 11005
Data columns (total 42 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   Unnamed: 0                 11006 non-null  int64              
 1   transaction_id             11006 non-null  object             
 2   customer_id                11006 non-null  object             
 3   timestamp                  10974 non-null  datetime64[ns, UTC]
 4   home_country               10982 non-null  object             
 5   source_currency            11006 non-null  object             
 6   dest_currency              11006 non-null  object             
 7   channel                    11006 non-null  object             
 8   amount_src                 11006 non-null  object             
 9   amount_usd                 11006 non-null  float64            
 10  fee                        11006 non-null  float64            
 11  ex

In [20]:
from pathlib import Path

artifacts_dir = Path("../Finlora_dataset/artifacts")

artifacts_dir.mkdir(parents=True, exist_ok=True)

print("Artifacts directory:", artifacts_dir.resolve())

Artifacts directory: C:\Users\user\Downloads\Fraudulent_Transaction_Detection_for_Finlora_Company\Finlora_dataset\artifacts


In [23]:
cleaned_customer_transaction_data.to_csv("../Finlora_Dataset/artifacts/Engineered_Data.csv", index=False)
